# Clash Royale Meta Analysis (April 2026)

### Data Analytics Project

- Analyzes deck and card performance using public Clash Royale data from RoyaleAPI
- Explores usage trends, win rates, rankings, and hero card performance
- Uses Python, SQL, web scraping, and data visualization techniques
- Demonstrates end-to-end analytics workflow from raw HTML data to insights

## Setup and Helper Functions

- Imports the libraries used for data processing, parsing, and visualization
- Defines reusable helper functions for reading files and cleaning raw values
- Extracts card information from deck URLs and formats it for analysis
- Prepares consistent metrics for tables and visualizations
- Reduces repetitive code and improves notebook organization

In [ ]:
import json
import re
from pathlib import Path
from urllib.parse import urlparse, unquote

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bs4 import BeautifulSoup
from IPython.display import display
from scipy.stats import gaussian_kde

def resolve_path(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path("/mnt/data") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename}")

def read_html(filename: str) -> BeautifulSoup:
    path = resolve_path(filename)
    with open(path, "r", encoding="utf-8") as f:
        return BeautifulSoup(f.read(), "html.parser")

def to_float(value):
    if value is None:
        return None
    text = str(value).replace("%", "").replace(",", "").strip()
    try:
        return float(text)
    except Exception:
        return None

def to_int(value):
    if value is None:
        return None
    text = str(value).replace(",", "").strip()
    try:
        return int(text)
    except Exception:
        return None

def extract_card_slugs_from_url(deck_url: str):
    path = unquote(urlparse(deck_url).path)
    slug_blob = path.split("/decks/stats/")[-1]
    return [slug.strip() for slug in slug_blob.split(",") if slug.strip()]

def slug_to_name(slug: str):
    clean = re.sub(r"-ev\d+$", "", slug)
    return " ".join(part.capitalize() for part in clean.split("-"))

def add_card_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["is_evo"] = pd.to_numeric(df["is_evo"], errors="coerce").fillna(0).astype(int)
    df["is_hero"] = pd.to_numeric(df["is_hero"], errors="coerce").fillna(0).astype(int)

    numeric_cols = [
        "usage_rate",
        "usage_delta",
        "win_rate",
        "win_delta",
        "rating_score",
        "clean_win_rate",
        "rank",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["usage_pct"] = df["usage_rate"] * 100
    df["win_pct"] = df["win_rate"]
    df["clean_win_pct"] = df["clean_win_rate"] * 100
    return df

def make_usage_share_pie(df, filter_col, filter_value, title, center_label):
    plot_df = (
        df[df[filter_col] == filter_value]
        .dropna(subset=["usage_pct", "win_pct"])
        .sort_values("usage_pct", ascending=False)
        .copy()
    )

    fig = px.pie(
        plot_df,
        names="card_name",
        values="usage_pct",
        hole=0.42,
        title=title
    )

    fig.update_traces(
        text=plot_df["card_name"] + "<br>" + plot_df["usage_pct"].round(1).astype(str) + "%",
        textinfo="text",
        textposition="inside",
        pull=[0.03] * len(plot_df),
        customdata=plot_df[["win_pct"]],
        hovertemplate="<b>%{label}</b><br>Win Rate: %{customdata[0]:.2f}%<extra></extra>"
    )

    fig.update_layout(
        template="plotly_white",
        width=900,
        height=720,
        title=dict(x=0.5, xanchor="center", font=dict(size=24, family="Arial")),
        font=dict(family="Arial", size=13),
        margin=dict(l=40, r=40, t=90, b=40),
        legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02),
        annotations=[dict(text=center_label, x=0.5, y=0.5, showarrow=False, font=dict(size=18))]
    )
    return fig

def make_rank_table(df, limit=None):
    columns = ["rank", "card_name", "usage_pct", "win_rate", "clean_win_pct", "rating_score"]
    table_df = df[columns]
    if limit is not None:
        table_df = table_df.head(limit)
    return table_df


## Deck Data Extraction and Structuring

- Extracts deck-level data from the RoyaleAPI HTML source
- Uses embedded JSON-LD metadata to recover deck names and rankings
- Parses deck segments to collect metrics such as average elixir, cycle cost, rating, usage, and win-related statistics
- Organizes the extracted values into a structured dataframe
- Produces a cleaned deck-level dataset for comparison and exploratory analysis

In [17]:

soup = read_html("royaleapi_popular.html")

scripts = soup.find_all("script", type="application/ld+json")
json_ld_text = next(
    (
        (tag.string if tag.string else tag.get_text())
        for tag in scripts
        if '"itemListElement"' in (tag.string if tag.string else tag.get_text())
        and '"ItemList"' in (tag.string if tag.string else tag.get_text())
    ),
    None,
)

data = json.loads(json_ld_text)
item_lists = [item for item in data.get("mainEntity", []) if item.get("@type") == "ItemList"]
deck_items = item_lists[0]["itemListElement"]

deck_lookup = {
    entry["item"]["url"]: {
        "rank": entry.get("position"),
        "deck_name": entry["item"].get("name"),
    }
    for entry in deck_items
}

deck_segments = soup.select("div.ui.attached.segment.deck_segment")

rows = []
for seg in deck_segments:
    deck_link_tag = seg.select_one('a[href*="/decks/stats/"]')
    if not deck_link_tag:
        continue

    href = deck_link_tag.get("href")
    deck_url = f"https://royaleapi.com{href}" if href.startswith("/") else href
    lookup = deck_lookup.get(deck_url, {})

    deck_name = lookup.get("deck_name")
    rank = lookup.get("rank")

    if not deck_name:
        title_tag = seg.select_one("h4.deck_human_name-mobile")
        deck_name = title_tag.get_text(strip=True) if title_tag else None

    seg_text = seg.get_text(" ", strip=True)
    avg_elixir_match = re.search(r"Avg Elixir\s*([0-9.]+)", seg_text)
    cycle_match = re.search(r"4-Card Cycle\s*([0-9.]+)", seg_text)

    avg_elixir = to_float(avg_elixir_match.group(1)) if avg_elixir_match else None
    four_card_cycle = to_float(cycle_match.group(1)) if cycle_match else None

    rating = usage_pct = win_rate = draw_rate = loss_rate = None
    usage_count = wins = draws = losses = total_battles = None

    stats_tables = seg.select("table.ui.very.basic.unstackable.compact.stats.table")
    chosen_table = None
    for tbl in stats_tables:
        rows_tr = tbl.select("tbody tr")
        if rows_tr:
            chosen_table = tbl
            if len(rows_tr) >= 2:
                break

    if chosen_table:
        rows_tr = chosen_table.select("tbody tr")

        first_row = rows_tr[0].find_all("td")
        if len(first_row) >= 5:
            rating = to_int(first_row[0].get_text(" ", strip=True))
            usage_pct = to_float(first_row[1].get_text(" ", strip=True))
            win_rate = to_float(first_row[2].get_text(" ", strip=True))
            draw_rate = to_float(first_row[3].get_text(" ", strip=True))
            loss_rate = to_float(first_row[4].get_text(" ", strip=True))

        if len(rows_tr) >= 2:
            second_row = rows_tr[1].find_all("td")
            if len(second_row) >= 5:
                usage_count = to_int(second_row[1].get_text(" ", strip=True))
                wins = to_int(second_row[2].get_text(" ", strip=True))
                draws = to_int(second_row[3].get_text(" ", strip=True))
                losses = to_int(second_row[4].get_text(" ", strip=True))

        if usage_count is not None and wins is None and win_rate is not None:
            wins = round(usage_count * win_rate / 100)
            draws = round(usage_count * (draw_rate or 0) / 100)
            losses = round(usage_count * (loss_rate or 0) / 100)

    if wins is not None and draws is not None and losses is not None:
        total_battles = wins + draws + losses

    card_slugs = extract_card_slugs_from_url(deck_url)
    card_names = [slug_to_name(slug) for slug in card_slugs]

    row = {
        "rank": rank,
        "deck_name": deck_name,
        "deck_url": deck_url,
        "avg_elixir": avg_elixir,
        "four_card_cycle": four_card_cycle,
        "rating": rating,
        "usage_pct": usage_pct,
        "win_rate": win_rate,
        "draw_rate": draw_rate,
        "loss_rate": loss_rate,
        "usage_count": usage_count,
        "wins": wins,
        "draws": draws,
        "losses": losses,
        "total_battles": total_battles,
        "card_slugs": card_slugs,
        "num_cards": len(card_slugs),
    }

    for i in range(8):
        row[f"card_{i+1}_slug"] = card_slugs[i] if i < len(card_slugs) else None
        row[f"card_{i+1}"] = card_names[i] if i < len(card_names) else None

    rows.append(row)

decks_df = pd.DataFrame(rows)
decks_df["rank"] = pd.to_numeric(decks_df["rank"], errors="coerce")
decks_df = decks_df.sort_values("rank").reset_index(drop=True)

display(
    decks_df[
        [
            "rank", "deck_name", "avg_elixir", "four_card_cycle",
            "rating", "usage_pct", "win_rate", "wins", "losses", "total_battles"
        ]
    ].head(10)
)


,rank,deck_name,avg_elixir,four_card_cycle,rating,usage_pct,win_rate,wins,losses,total_battles
0,1,HeroGobs EvoGhost Bait,3.3,9.0,53,0.4,54.7,777,643,1421
1,2,EvoLJ GS HeroWiz Zappies,4.1,14.0,53,2.2,52.5,4311,3900,8215
2,3,EvoMortar Miner Cart HeroGobs,3.9,12.0,51,0.6,53.0,1216,1072,2294
3,4,Loon HeroKnight Double Dragon Bowler,3.8,12.0,50,0.6,52.2,1244,1124,2385
4,5,GS Berserker GobHut Evo Bats,3.6,10.0,50,1.2,51.6,2244,2086,4350
5,6,EvoMortar Cart HeroGobs Bait,3.5,10.0,50,1.9,51.3,3646,3452,7103
6,7,Evo GobDrill HeroKnight 2.6 Cycle,2.6,6.0,49,0.4,52.9,709,625,1340
7,8,GK GY Bowler Evo Exec,4.3,15.0,49,1.0,51.4,1964,1843,3819
8,9,X-Bow HeroKnight 3.0 Cycle,3.0,7.0,49,2.7,50.6,5101,4953,10072
9,10,Hog Evo Exec HeroGobs Nado,3.4,8.0,48,1.1,51.3,2049,1936,3994


## Card-Level Data Extraction

- Parsed card statistics from RoyaleAPI HTML source
- Extracted usage rate, win rate, rank, rarity, and card type
- Cleaned raw values and converted fields into numeric format
- Built a structured dataframe for further analysis

In [18]:

cards_soup = read_html("cards_stat.html")
card_nodes = cards_soup.select("div.grid_item[data-card]")

card_rows = []
for node in card_nodes:
    card_rows.append(
        {
            "card_slug": node.get("data-card"),
            "card_name": node.select_one(".card_name").get_text(strip=True) if node.select_one(".card_name") else None,
            "card_type": node.get("data-type"),
            "rarity": node.get("data-rarity"),
            "is_evo": to_int(node.get("data-evo")),
            "is_hero": to_int(node.get("data-hero")),
            "usage_rate": to_float(node.get("data-usage")),
            "usage_delta": to_float(node.get("data-delta")),
            "win_rate": to_float(node.get("data-winpercent")),
            "win_delta": to_float(node.get("data-windelta")),
            "rating_score": to_float(node.get("data-rating")),
            "clean_win_rate": to_float(node.get("data-cleanwinrate")),
            "rank": to_int(
                node.select_one(".card_rank_label").get_text(strip=True)
                if node.select_one(".card_rank_label") else None
            ),
        }
    )

cards_df = pd.DataFrame(card_rows).sort_values("rank").reset_index(drop=True)
cards_df = add_card_metrics(cards_df)
display(cards_df.head())


,card_slug,card_name,card_type,rarity,is_evo,is_hero,usage_rate,usage_delta,win_rate,win_delta,rating_score,clean_win_rate,rank,usage_pct,win_pct,clean_win_pct
0,zappies,Zappies,Troop,Rare,0,0,0.070950,-0.09,53.133448,-0.592643,0.573192,0.533728,1,7.095006,53.133448,53.372771
1,cannon-cart,Cannon Cart,Troop,Epic,0,0,0.052152,0.22,53.070985,-1.015001,0.568211,0.532400,2,5.215187,53.070985,53.239972
2,rascals,Rascals,Troop,Common,0,0,0.055653,0.22,52.915396,-0.599799,0.564750,0.530872,3,5.565260,52.915396,53.087217
3,battle-ram-ev1,Battle Ram Evolution,Troop,Rare,1,0,0.073195,0.32,52.726763,-0.469924,0.562697,0.529421,4,7.319495,52.726763,52.942135
4,bowler,Bowler,Troop,Epic,0,0,0.052914,0.04,52.504624,-0.079773,0.554494,0.526446,5,5.291437,52.504624,52.644594


## SQL Setup

- Creates a SQLite connection inside the notebook
- Loads the cleaned card dataset into a SQL table
- Enables SQL-based querying for analytics tasks

In [30]:
import sqlite3
conn = sqlite3.connect("clash_royale.db")
cards_df.to_sql("cards", conn, if_exists="replace", index=False)

176

## SQL Query: Most Used Cards

- Ranks cards by usage rate
- Identifies the most popular cards in the current meta

In [ ]:
#highest usage rate cards
query = """
SELECT card_name, usage_rate
FROM cards
ORDER BY usage_rate DESC
LIMIT 10
"""
pd.read_sql(query, conn)

## SQL Query: Balance Risk Cards

- Filters cards with high usage and high win rate
- Highlights potential overperforming cards

In [ ]:
#balance risk cards(high wr with eough usage rate)
query = """
SELECT card_name, usage_rate, win_rate
FROM cards
WHERE usage_rate*100 > 4 AND win_rate > 52
ORDER BY win_rate DESC
"""
pd.read_sql(query, conn)

## SQL Query: Hero Performance

- Reviews hero cards by win rate and usage rate
- Compares hero impact within the meta

In [42]:
query = """
SELECT 
    card_name,
    ROUND(win_rate, 2) AS win_rate,
    ROUND(usage_rate, 2) AS usage_rate
FROM cards
WHERE LOWER(card_name) LIKE '%hero%'
ORDER BY win_rate DESC, usage_rate DESC
"""

hero_stats = pd.read_sql(query, conn)
hero_stats

,card_name,win_rate,usage_rate
0,Hero Giant,51.96,0.03
1,Hero Goblins,51.80,0.10
2,Hero Barbarian Barrel,51.76,0.12
3,Hero Balloon,51.14,0.08
4,Hero Mega Minion,50.63,0.06
5,Hero Wizard,50.52,0.05
6,Hero Ice Golem,49.50,0.04
7,Hero Knight,49.39,0.13
8,Hero Mini P.E.K.K.A,48.29,0.06
9,Hero Magic Archer,47.78,0.08


## Hero Rankings

- Displays top hero cards using rank, usage, and win metrics

In [19]:

strongest_heroes = cards_df[cards_df["is_hero"] == 1].sort_values("rank").reset_index(drop=True)
make_rank_table(strongest_heroes)


,rank,card_name,usage_pct,win_rate,clean_win_pct,rating_score
0,14,Hero Barbarian Barrel,11.804619,51.760615,51.996270,0.541842
1,15,Hero Goblins,9.559334,51.803807,51.994011,0.540581
2,18,Hero Giant,3.132269,51.960587,52.022685,0.534969
3,25,Hero Balloon,7.720323,51.135399,51.230403,0.520598
4,40,Hero Mega Minion,6.013843,50.631333,50.671731,0.505166
5,45,Hero Wizard,5.478985,50.524405,50.554810,0.501587
6,89,Hero Knight,12.832501,49.386118,49.295387,0.474584
7,92,Hero Ice Golem,4.231034,49.503750,49.481825,0.473677
8,127,Hero Mini P.E.K.K.A,6.416724,48.291345,48.173513,0.443268
9,139,Hero Magic Archer,8.064713,47.783107,47.588630,0.429881


## Evolution Rankings

- Displays top evolution cards using rank, usage, and win metrics

In [20]:

strongest_evolutions = cards_df[cards_df["is_evo"] == 1].sort_values("rank").reset_index(drop=True)
make_rank_table(strongest_evolutions)


,rank,card_name,usage_pct,win_rate,clean_win_pct,rating_score
0,4,Battle Ram Evolution,7.319495,52.726763,52.942135,0.562697
1,7,Mortar Evolution,4.939469,52.453777,52.580439,0.551642
2,8,Zap Evolution,6.084648,52.337031,52.487775,0.550891
3,12,Lumberjack Evolution,9.618929,51.935007,52.140950,0.544079
4,16,Inferno Dragon Evolution,6.470162,51.839110,51.966347,0.538523
5,19,Goblin Cage Evolution,6.209127,51.592891,51.697684,0.531127
6,20,Royal Ghost Evolution,8.268285,51.498344,51.632878,0.530600
7,21,Skeleton Barrel Evolution,7.758764,51.407003,51.524811,0.527741
8,29,Minion Horde Evolution,2.681949,51.222886,51.256616,0.515131
9,32,Archers Evolution,3.465846,51.031703,51.068749,0.512156


## Overall Card Rankings

- Summarizes the strongest cards across the dataset
- Compares rating, usage, and win performance

In [21]:

strongest_cards = cards_df.sort_values("rank").reset_index(drop=True)
make_rank_table(strongest_cards, limit=20)


,rank,card_name,usage_pct,win_rate,clean_win_pct,rating_score
0,1,Zappies,7.095006,53.133448,53.372771,0.573192
1,2,Cannon Cart,5.215187,53.070985,53.239972,0.568211
2,3,Rascals,5.565260,52.915396,53.087217,0.564750
3,4,Battle Ram Evolution,7.319495,52.726763,52.942135,0.562697
4,5,Bowler,5.291437,52.504624,52.644594,0.554494
5,6,Dark Prince,10.127187,52.273206,52.529379,0.554139
6,7,Mortar Evolution,4.939469,52.453777,52.580439,0.551642
7,8,Zap Evolution,6.084648,52.337031,52.487775,0.550891
8,9,Lava Hound,4.831408,52.351000,52.470386,0.548900
9,10,Skeleton Dragons,9.020844,52.046236,52.249142,0.546864


## Visualization: Top Card Ratings

- Compares leading cards by rating score
- Uses a visual ranking format for quick comparison

In [22]:

plot_df = strongest_cards.head(15).copy()
plot_df["rating_scaled"] = plot_df["rating_score"] * 100
plot_df = plot_df.sort_values("rating_scaled", ascending=True)

fig = px.bar(
    plot_df,
    x="rating_scaled",
    y="card_name",
    orientation="h",
    text="rating_scaled",
    color="rating_scaled",
    color_continuous_scale="Viridis",
    title="Top 15 Strongest Cards in the Current Meta"
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    marker_line_width=1,
    marker_line_color="rgba(255,255,255,0.25)"
)

fig.update_layout(
    template="plotly_dark",
    height=720,
    width=1100,
    title={"x": 0.5, "xanchor": "center", "font": {"size": 24}},
    font={"family": "Arial", "size": 14},
    margin=dict(l=40, r=40, t=80, b=40),
    coloraxis_showscale=False,
    xaxis=dict(title="Rating Score (0–100)", showgrid=True, gridcolor="rgba(255,255,255,0.08)"),
    yaxis=dict(title="", categoryorder="total ascending", showgrid=False)
)

fig.show()


## Deck Composition Analysis

- Counts card appearances across popular decks
- Identifies frequently used cards in top decks

In [23]:

all_cards = []
for i in range(1, 9):
    col = f"card_{i}"
    if col in decks_df.columns:
        all_cards.extend(decks_df[col].dropna().tolist())

card_counts = pd.Series(all_cards).value_counts().reset_index()
card_counts.columns = ["card_name", "count"]
card_counts["percentage"] = card_counts["count"] / card_counts["count"].sum() * 100

top_n = 20
if len(card_counts) > top_n:
    top_cards = card_counts.head(top_n).copy()
    others = pd.DataFrame(
        {
            "card_name": ["Others"],
            "count": [card_counts.iloc[top_n:]["count"].sum()],
            "percentage": [card_counts.iloc[top_n:]["percentage"].sum()],
        }
    )
    plot_df = pd.concat([top_cards, others], ignore_index=True)
else:
    plot_df = card_counts.copy()

fig = px.pie(
    plot_df,
    names="card_name",
    values="count",
    hole=0.38,
    title="Card Usage Share Across Top Meta Decks"
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>",
    pull=[0.03] * len(plot_df)
)

fig.update_layout(
    template="plotly_white",
    height=760,
    width=980,
    title={"x": 0.5, "xanchor": "center", "font": {"size": 24, "family": "Arial"}},
    font={"family": "Arial", "size": 13},
    margin=dict(l=40, r=40, t=90, b=40),
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02),
    annotations=[dict(text="Top Meta Decks", x=0.5, y=0.5, font_size=18, showarrow=False)]
)

fig.show()


## Visualization: Evolution Usage Share

- Shows usage distribution among evolution cards
- Uses win rate in hover details for added context

In [24]:

fig = make_usage_share_pie(
    cards_df,
    filter_col="is_evo",
    filter_value=1,
    title="Evolution Usage Share in the Current Meta",
    center_label="Evolutions",
)
fig.show()


## Visualization: Hero Usage Share

- Shows usage distribution among hero cards
- Highlights relative popularity, showcasing the 3 highly-used heroes

In [26]:

hero_df = (
    cards_df[cards_df["is_hero"] == 1]
    .dropna(subset=["usage_pct"])
    .sort_values("usage_pct", ascending=False)
    .copy()
)

equal_share = hero_df["usage_pct"].sum() / len(hero_df)
hero_df = hero_df.sort_values("usage_pct", ascending=True)

fig = px.bar(
    hero_df,
    x="usage_pct",
    y="card_name",
    orientation="h",
    text="usage_pct",
    color="usage_pct",
    color_continuous_scale="Blues",
    title="Hero Usage Rate in Current Meta"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    marker_line_width=1.1,
    marker_line_color="rgba(255,255,255,0.25)",
    hovertemplate="<b>%{y}</b><br>Usage Rate: %{x:.2f}%<extra></extra>"
)

fig.add_vline(
    x=equal_share,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"Ideal Equal Share = {equal_share:.1f}%",
    annotation_position="top"
)

fig.update_layout(
    template="plotly_white",
    width=1050,
    height=700,
    title=dict(x=0.5, xanchor="center", font=dict(size=24, family="Arial")),
    font=dict(family="Arial", size=14),
    margin=dict(l=70, r=50, t=90, b=50),
    coloraxis_showscale=False,
    xaxis=dict(title="Usage Rate (%)", showgrid=True, gridcolor="rgba(0,0,0,0.08)", zeroline=False),
    yaxis=dict(title="", showgrid=False, tickfont=dict(size=13))
)

fig.show()


## Visualization: Deck Elixir vs Usage

- Examines relationship between average elixir cost and deck popularity
- Helps assess efficiency trends
- 2 peaks at 2.8 and 4 elixir suggest a hyperbait and bridgespam meta 

In [27]:

plot_df = decks_df.copy()
plot_df["avg_elixir"] = pd.to_numeric(plot_df["avg_elixir"], errors="coerce")
plot_df["usage_count"] = pd.to_numeric(plot_df["usage_count"], errors="coerce")
plot_df = plot_df.dropna(subset=["avg_elixir", "usage_count"]).copy()

x = plot_df["avg_elixir"].to_numpy()
weights = plot_df["usage_count"].to_numpy()

bins = np.linspace(x.min(), x.max(), 12)
hist, edges = np.histogram(x, bins=bins, weights=weights)
centers = (edges[:-1] + edges[1:]) / 2
bin_width = np.diff(edges)

kde = gaussian_kde(x, weights=weights)
x_grid = np.linspace(x.min(), x.max(), 300)
y_kde = kde(x_grid)

scale_factor = hist.max() / y_kde.max() if y_kde.max() > 0 else 1
y_kde_scaled = y_kde * scale_factor

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=centers,
        y=hist,
        width=bin_width,
        name="Usage Count",
        opacity=0.55,
        hovertemplate="Avg Elixir Bin: %{x:.2f}<br>Total Usage: %{y:,.0f}<extra></extra>"
    )
)

fig.add_trace(
    go.Scatter(
        x=x_grid,
        y=y_kde_scaled,
        mode="lines",
        name="Smoothed Trend",
        line=dict(width=4),
        hovertemplate="Avg Elixir: %{x:.2f}<br>Smoothed Usage Trend: %{y:,.0f}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    width=1120,
    height=700,
    title=dict(
        text="Distribution of Deck Usage by Average Elixir Cost",
        x=0.5,
        xanchor="center",
        font=dict(size=24, family="Arial")
    ),
    font=dict(family="Arial", size=14),
    margin=dict(l=60, r=40, t=90, b=55),
    barmode="overlay",
    xaxis=dict(title="Average Elixir Cost", showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=False),
    yaxis=dict(title="Total Usage Count", showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()


## Visualization: Usage vs Win Rate

- Compares popularity against performance
- Identifies strong and widely used cards

In [28]:

plot_df = cards_df.copy()
plot_df = plot_df.dropna(subset=["usage_pct", "win_pct", "rating_score"]).copy()
plot_df = plot_df[plot_df["usage_pct"] >= 2.0].copy()

usage_median = plot_df["usage_pct"].median()
win_baseline = 50

top_usage = plot_df.nlargest(8, "usage_pct")
top_win = plot_df.nlargest(8, "win_pct")
top_rating = plot_df.nlargest(8, "rating_score")
top_right = plot_df[
    (plot_df["usage_pct"] >= usage_median) & (plot_df["win_pct"] >= win_baseline)
].nlargest(10, "rating_score")

highlight_df = pd.concat([top_usage, top_win, top_rating, top_right]).drop_duplicates(subset=["card_name"])
base_df = plot_df[~plot_df["card_name"].isin(highlight_df["card_name"])].copy()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=base_df["usage_pct"],
        y=base_df["win_pct"],
        mode="markers",
        name="All cards",
        customdata=base_df[["card_name", "rating_score"]],
        marker=dict(
            size=10,
            color=base_df["rating_score"],
            colorscale="Viridis",
            opacity=0.45,
            line=dict(width=0.6, color="white"),
            showscale=True,
            colorbar=dict(title="Rating")
        ),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Usage: %{x:.2f}%<br>"
            "Win Rate: %{y:.2f}%<br>"
            "Rating: %{customdata[1]:.3f}<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=highlight_df["usage_pct"],
        y=highlight_df["win_pct"],
        mode="markers+text",
        name="Highlighted cards",
        text=highlight_df["card_name"],
        textposition="top center",
        customdata=highlight_df[["card_name", "rating_score"]],
        marker=dict(
            size=16,
            color=highlight_df["rating_score"],
            colorscale="Viridis",
            opacity=0.95,
            line=dict(width=1.2, color="white"),
            showscale=False
        ),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Usage: %{x:.2f}%<br>"
            "Win Rate: %{y:.2f}%<br>"
            "Rating: %{customdata[1]:.3f}<extra></extra>"
        )
    )
)

fig.add_hline(y=win_baseline, line_dash="dash", line_color="gray", opacity=0.7)
fig.add_vline(x=usage_median, line_dash="dash", line_color="gray", opacity=0.7)

fig.update_layout(
    template="plotly_white",
    width=1180,
    height=780,
    title=dict(text="Card Usage vs Win Rate", x=0.5, xanchor="center", font=dict(size=26, family="Arial")),
    font=dict(family="Arial", size=14),
    margin=dict(l=70, r=60, t=90, b=60),
    xaxis=dict(title="Usage Rate (%)", showgrid=True, gridcolor="rgba(0,0,0,0.08)", zeroline=False),
    yaxis=dict(title="Win Rate (%)", showgrid=True, gridcolor="rgba(0,0,0,0.08)", zeroline=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()
